# Projet final : analyse et visualisation de donnees

Ce notebook illustre une demarche de bout en bout : donnees brutes, preparation, indicateurs, puis visualisations — comme base pour un eventuel outil interactif.

Les etapes abordees sont les suivantes :

- prise en main et comprehension du jeu de donnees ;
- chargement, controles de base et nettoyage ;
- construction de variables derivees et d'indicateurs metiers ;
- graphiques statiques et interactifs pour synthetiser les resultats.


In [ ]:
# Bibliotheques necessaires (programme du cours)
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio

In [ ]:
# Affichage Plotly dans le navigateur (souvent plus fiable selon l'environnement)
pio.renderers.default = 'browser'

## 1. Exploration et preparation (pandas)

But : preparer un jeu de donnees propre et exploitable pour la suite.

On commence par le chargement et un premier examen des fichiers.


In [ ]:
# Chargement du fichier CSV
df = pd.read_csv('Dataset.csv')

# Apercu des premieres lignes
df.head()

In [ ]:
# Dimensions : (nombre de lignes, nombre de colonnes)
df.shape

In [ ]:
# Types de colonnes et comptage des valeurs non nulles
df.info()

In [ ]:
# Bilan des valeurs manquantes par colonne
df.isnull().sum()

## 2. Nettoyage, manipulation et preparation des variables

Les horodatages sont convertis au format adapte ; on en deduit des decoupes temporelles (jour, heure, etc.), puis des indicateurs financiers (montants, marges, rentabilite).


In [ ]:
# Typage datetime pour la colonne horodatage
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Controle apres conversion
df.head()

In [ ]:
# Granularites temporelles derivees (jour, heure, mois, etc.)
df['Date'] = df['TransactionStartTime'].dt.date
df['Hour'] = df['TransactionStartTime'].dt.hour
df['Day'] = df['TransactionStartTime'].dt.day
df['Month'] = df['TransactionStartTime'].dt.month

df.head()

In [ ]:
# Montant en valeur absolue, marge brute et taux de rentabilite (eviter division par zero)
df['AmountAbs'] = df['Amount'].abs()
df['MargeBrute'] = df['Value'] - df['AmountAbs']
df['TauxRentabilite'] = (df['MargeBrute'] / df['AmountAbs'].replace(0, pd.NA)) * 100

df[['Amount', 'AmountAbs', 'Value', 'MargeBrute', 'TauxRentabilite']].head()

## 3. Analyse avec pandas

On etablit des effectifs, des moyennes et des resumes numeriques par dimension : cela fixe les ordres de grandeur avant la partie graphique.


In [ ]:
# Effectifs par categorie de produit
df['ProductCategory'].value_counts()

In [ ]:
# Repartition des transactions par canal
df['ChannelId'].value_counts()

In [ ]:
# Balance fraude / non-fraude
df['FraudResult'].value_counts()

In [ ]:
# Resume numerique (moyenne, ecarts, quantiles) des variables cles
df[['Amount', 'Value', 'AmountAbs', 'MargeBrute', 'TauxRentabilite']].describe()

In [ ]:
# Proportion de fraude par categorie (moyenne de l'indicateur binaire)
fraud_rate_category = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False)
fraud_rate_category

In [ ]:
# Valeur transaction moyenne par categorie
df.groupby('ProductCategory')['Value'].mean().sort_values(ascending=False)

In [ ]:
# Synthese par strategie de prix : moyenne, somme et nombre d'observations
df.groupby('PricingStrategy')[['AmountAbs', 'Value', 'MargeBrute']].agg(['mean', 'sum', 'count']).round(2)

## 4. Analyse graphique

Les figures rendent visibles les tendances, les concentrations et les ecarts entre categories ou canaux.

- **Sorties statiques** (matplotlib, seaborn) : lecture directe, pratique pour un compte rendu ou une capture d'ecran.
- **Sorties interactives** (plotly express) : inspection au survol (valeurs, legendes) et navigation dans le detail.


In [ ]:
# Volume journalier de transactions
transactions_per_day = df.groupby('Date').size()

In [ ]:
# Serie temporelle : activite par jour (matplotlib)
plt.figure(figsize=(14, 5))
transactions_per_day.plot()
plt.title('Nombre de transactions par jour')
plt.xlabel('Date')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
# Histogramme des effectifs par heure de la journee (seaborn)
plt.figure(figsize=(10, 5))
sns.countplot(data=df, x='Hour', color='skyblue')
plt.title('Repartition des transactions par heure')
plt.xlabel('Heure')
plt.ylabel('Nombre de transactions')
plt.show()

In [ ]:
# Tableau agrege pour le camembert par canal
channel_counts = df['ChannelId'].value_counts().reset_index()
channel_counts.columns = ['ChannelId', 'Count']

In [ ]:
# Part du trafic par canal (plotly)
fig = px.pie(channel_counts, names='ChannelId', values='Count',
             title='Repartition des transactions par canal')
fig.show(renderer='browser')

In [ ]:
# Rentabilite moyenne par categorie (tri croissant pour le barh)
rentabilite = df.groupby('ProductCategory')['TauxRentabilite'].mean().sort_values()

In [ ]:
# Comparaison des rentabilites moyennes entre categories (barres horizontales)
plt.figure(figsize=(12, 5))
rentabilite.plot(kind='barh', color='teal')
plt.title('Taux de rentabilite moyen par categorie')
plt.xlabel('Taux de rentabilite')
plt.show()

In [ ]:
# Matrice de correlation entre variables numeriques (dont la cible fraude)
plt.figure(figsize=(8, 5))
sns.heatmap(df[['Amount', 'Value', 'MargeBrute', 'TauxRentabilite', 'FraudResult']].corr(), annot=True, cmap='coolwarm')
plt.title('Correlations entre variables')
plt.show()

In [ ]:
# Chiffre d'affaires cumule par categorie (somme des Value)
revenue_per_category = df.groupby('ProductCategory')['Value'].sum().sort_values(ascending=False).reset_index()

In [ ]:
# Barres : contribution au revenu par categorie (plotly)
fig = px.bar(revenue_per_category,
             x='ProductCategory', y='Value',
             title='Revenu total par categorie de produit',
             color='Value')
fig.show(renderer='browser')

In [ ]:
# Fraude en pourcentage par categorie (plotly)
fraud_plot = df.groupby('ProductCategory')['FraudResult'].mean().sort_values(ascending=False).reset_index()
fraud_plot['FraudResult'] = fraud_plot['FraudResult'] * 100

fig = px.bar(fraud_plot, x='ProductCategory', y='FraudResult', color='FraudResult',
             title='Taux de fraude par categorie (%)')
fig.show(renderer='browser')

## 5. Conclusion

En suivant le fil habituel d'un mini-projet d'analyse — donnees brutes, preparation, chiffres cles, puis figures — on obtient une lecture coherente du jeu de donnees.

Points saillants :

- les cas de fraude restent peu nombreux par rapport au volume total ;
- la categorie `financial_services` domine en nombre de transactions ;
- `ChannelId_3` et `ChannelId_2` expliquent l'essentiel du trafic observe ;
- les visualisations confirment des variations dans le temps et des profils differents selon le canal et la categorie de produit.

Une **application Streamlit** peut ensuite reprendre ces elements sous forme de tableau de bord filtrable pour un usage plus interactif au quotidien.
